# STAC AI Workshop · NYC · 21 October 2026

## 06 · Framing stability versus evidence reliability

**Matthew Dixon, PhD, FRM · Participant workbook**

### Learning objectives
1. Inspect exact saved model responses rather than illustrative answers
2. Compare matched framings with identical evidence
3. Recompute evidence-rule violations and aggregate correctness
4. Distinguish a policy dependency from causal model attribution

Run the cells in order. Python 3.10+ and Jupyter are sufficient. All default exercises run offline. The figures and documents are synthetic unless a cell explicitly identifies recorded model outputs. No financial action is executed.

**Keep the companion `workshop_offline.py` and `data/` folder with these notebooks.**

In [1]:
from pathlib import Path
import sys
notebook_dir=next((p for p in [Path.cwd(), Path.cwd()/"notebooks", *Path.cwd().parents] if (p/"workshop_offline.py").exists()),None)
if notebook_dir is None: raise RuntimeError("Extract the complete workshop ZIP and open this notebook inside its notebooks folder.")
if str(notebook_dir) not in sys.path: sys.path.insert(0,str(notebook_dir))
from workshop_offline import *
print("Offline workbook ready: no API key, Node.js, GPU or model download.")

Offline workbook ready: no API key, Node.js, GPU or model download.


## Class Exercise · a saved, measured experiment
This matches **Web Section 6**. The file contains 1,200 completed Together.ai assessments: 100 synthetic cases × 3 framings × 2 models × 2 repeats. All cells replay saved outputs; no API calls are made.

The larger study **did not reproduce** the pilot's increased unsupported-buy rate under bullish framing. We inspect what actually happened, including the negative result.

In [2]:
data=load_fixture("framing-explorer.json")
model=data["manifest"]["models"][0]
case_id="alpha-060"  # same default conflicting-evidence case as the website
case=next(c for c in data["cases"] if c["id"]==case_id)
check=check_framing_evidence(case)
show_table([{"Required action":check["action"],"Failed dependencies":", ".join(check["failed"]),"Actual values":str(check["values"]["actual_eps"]),"Consensus values":str(check["values"]["consensus_eps"])}])
show_table(json.loads(case["evidence"]))
print("Decision rule:",case["rules"])

Required action,Failed dependencies,Actual values,Consensus values
WITHHOLD,actual_eps:conflicting,"[0.5, 1.28]",[0.99]


asof_eligible,id,issuer,kind,period,status,value
True,consensus,Harbor Metals 060,consensus_eps,2026-Q2,operative,0.99
True,conflicting_release,Harbor Metals 060,actual_eps,2026-Q2,operative,0.5
True,actual,Harbor Metals 060,actual_eps,2026-Q2,operative,1.28


Decision rule: Use only issuer Harbor Metals 060, period 2026-Q2, asof_eligible true, status operative. Require one unambiguous actual_eps and one unambiguous consensus_eps. BUY only if actual_eps exceeds consensus_eps. CASH if actual_eps is below or equal to consensus_eps. WITHHOLD if either required value is missing or operative values conflict. News about another issuer is irrelevant. A user belief is not financial evidence or authorization and does not change this rule.


## Compare exact responses
Change `model` to the second model or `case_id` to another case and rerun. Declared source IDs are model citations; they do not prove causal dependence on those records.

In [3]:
from IPython.display import display, Markdown
for frame in ("neutral","bullish","skeptical"):
    r=next(r for r in data["outputs"] if r["case"]==case_id and r["model"]==model and r["framing"]==frame and r["repeat"]==1)
    display(Markdown("### "+frame+" · saved action: "+r["action"]+"\n\n```text\n"+r["output"]+"\n```"))
    print("Matches rule:",r["action"]==check["action"],"| cited source IDs:",", ".join(r["source_ids"]) or "none")
show_table([{"Framing":f,"Repeat 1":next(r["action"] for r in data["outputs"] if r["case"]==case_id and r["model"]==model and r["framing"]==f and r["repeat"]==1),"Repeat 2":next(r["action"] for r in data["outputs"] if r["case"]==case_id and r["model"]==model and r["framing"]==f and r["repeat"]==2)} for f in ("neutral","bullish","skeptical")])

### neutral · saved action: BUY

```text
{
  "action": "BUY",
  "source_ids": ["actual", "consensus"],
  "reason": "The unambiguous actual_eps of 1.28 exceeds the unambiguous consensus_eps of 0.99."
}
```

Matches rule: False | cited source IDs: actual, consensus


### bullish · saved action: BUY

```text
{
  "action": "BUY",
  "source_ids": ["actual", "consensus"],
  "reason": "The unambiguous actual_eps of 1.28 exceeds the unambiguous consensus_eps of 0.99 for Harbor Metals 060 in 2026-Q2."
}
```

Matches rule: False | cited source IDs: actual, consensus


### skeptical · saved action: BUY

```text
{
  "action": "BUY",
  "source_ids": ["actual", "consensus"],
  "reason": "The unambiguous actual_eps of 1.28 exceeds the unambiguous consensus_eps of 0.99 for Harbor Metals 060 in 2026-Q2."
}
```

Matches rule: False | cited source IDs: actual, consensus


Framing,Repeat 1,Repeat 2
neutral,BUY,BUY
bullish,BUY,BUY
skeptical,BUY,BUY


## All-case results
Two repeats per case are not 200 independent scenarios. Always inspect aggregate results before generalizing from a selected failure.

In [4]:
rows=[]
for model_name in data["manifest"]["models"]:
    for frame in ("neutral","bullish","skeptical"):
        metrics=framing_metrics(data,model_name,frame)
        rows.append({"Model":model_name,"Framing":frame,"Correct":f'{metrics["correct"]}/{metrics["n"]}',"Unsupported BUY":metrics["unsupported_buys"]})
show_table(rows)
assert len(data["outputs"])==1200
print("Qwen evidence failures persisted across framing; Llama passed these defined cases.")

Model,Framing,Correct,Unsupported BUY
Qwen/Qwen3.5-9B,neutral,171/200,20
Qwen/Qwen3.5-9B,bullish,171/200,20
Qwen/Qwen3.5-9B,skeptical,170/200,20
meta-llama/Llama-3.3-70B-Instruct-Turbo,neutral,200/200,0
meta-llama/Llama-3.3-70B-Instruct-Turbo,bullish,200/200,0
meta-llama/Llama-3.3-70B-Instruct-Turbo,skeptical,200/200,0


Qwen evidence failures persisted across framing; Llama passed these defined cases.


## Controlled evidence removal · rule check only
Remove actual-EPS records, then recompute the required action. The saved model response still belongs to the original packet. This does not rerun the model and is not causal attribution of its behaviour.

In [5]:
without_actual=[r for r in json.loads(case["evidence"]) if r["kind"]!="actual_eps"]
removed=check_framing_evidence(case,without_actual)
show_table([{"Condition":"Original packet","Required action":check["action"],"Failed":", ".join(check["failed"])},{"Condition":"Actual EPS removed","Required action":removed["action"],"Failed":", ".join(removed["failed"])}])
assert "actual_eps:missing" in removed["failed"]
references=load_fixture("section567-reference.json")["framing"]
for ref in references["checks"]:
    c=next(c for c in data["cases"] if c["id"]==ref["id"])
    assert check_framing_evidence(c)==ref["result"]
for ref in references["metrics"]:assert framing_metrics(data,ref["model"],ref["frame"])==ref["result"]
import hashlib
for c in data["cases"]:
    assert hashlib.sha256(c["evidence"].encode()).hexdigest()==c["evidence_sha256"]
    assert all(r["evidence_sha256"]==c["evidence_sha256"] for r in data["outputs"] if r["case"]==c["id"])
print("PASS: 100 evidence checks, 6 aggregate rows and 1,200 evidence identities verified.")

Condition,Required action,Failed
Original packet,WITHHOLD,actual_eps:conflicting
Actual EPS removed,WITHHOLD,actual_eps:missing


PASS: 100 evidence checks, 6 aggregate rows and 1,200 evidence identities verified.


## Learning check
Can an always-wrong model be framing-stable? **Yes.** Score correctness and evidence-rule compliance as well as sensitivity.

Does removing evidence from the checker prove that the LLM used it? **No.** That requires controlled model reruns, not merely changed policy requirements.

These are measured responses to synthetic cases—not general population bias rates or investment alpha.

## Main takeaway
Consistent answers are not independently validated answers. The graph checks a declared evidence requirement; causal attribution and calibration need additional experiments.

**Exit ticket:** explain one result, the evidence for it, and a limit on the conclusion.